# NavRL：Colab/G4 训练、TensorBoard 和模型下载

这个 notebook 会克隆仓库、准备 Isaac Sim Python 环境、运行 headless 训练、在 TensorBoard 中查看指标、下载模型和日志，并提供 GitHub 同步及本地验证入口。

普通 Colab 运行时不预装 Isaac Sim。请使用预配置的 Isaac Sim 镜像，或把项目 README 要求的 Isaac Sim 2023.1.0-hotfix.1 放到 Google Drive/挂载路径，并填写 ISAACSIM_PATH。建议先用较小的 MAX_FRAME_NUM 做冒烟测试。

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = ""  # 例如：https://github.com/<user>/<repo>.git
BRANCH = "main"
WORKSPACE = Path("/content/NavRL")
ISAACSIM_PATH = Path("/content/isaac-sim-2023.1.0-hotfix.1")
DEVICE = "cuda:0"
NUM_ENVS = 2
MAX_FRAME_NUM = 100_000
EVAL_INTERVAL = 1_000
SAVE_INTERVAL = 1_000
RUN_TRAINING = True
PUSH_CHANGES = False

TRAIN_ROOT = WORKSPACE / "isaac-training"
TRAIN_SCRIPT = TRAIN_ROOT / "training" / "scripts" / "train.py"
CHECKPOINT_DIR = TRAIN_ROOT / "outputs" / "checkpoints"
TENSORBOARD_DIR = TRAIN_ROOT / "outputs" / "tensorboard"
TRAIN_LOG = TRAIN_ROOT / "outputs" / "colab_train.log"
ARTIFACT_ARCHIVE = Path("/content/NavRL_artifacts.zip")

def run(command, *, cwd=None, check=True, env=None):
    print("$", " ".join(map(str, command)))
    return subprocess.run(command, cwd=cwd, check=check, env=env)

run(["nvidia-smi", "-L"], check=False)
import torch
if not torch.cuda.is_available():
    raise RuntimeError("没有检测到 CUDA，请在 Colab 中选择 GPU runtime。")
print("GPU:", torch.cuda.get_device_name(0), "PyTorch:", torch.__version__)

## 1. 克隆或更新 GitHub 仓库

In [ ]:
if not REPO_URL:
    raise ValueError("请先填写 REPO_URL。")
if WORKSPACE.exists() and not (WORKSPACE / ".git").exists():
    raise RuntimeError(f"目标目录已存在但不是 Git 仓库：{WORKSPACE}")
if not WORKSPACE.exists():
    run(["git", "clone", "--branch", BRANCH, REPO_URL, str(WORKSPACE)])
else:
    run(["git", "-C", str(WORKSPACE), "fetch", "origin", BRANCH])
    run(["git", "-C", str(WORKSPACE), "checkout", BRANCH])
    run(["git", "-C", str(WORKSPACE), "pull", "--ff-only", "origin", BRANCH])
print("仓库路径：", WORKSPACE)

## 2. 创建/准备运行环境

训练依赖 Isaac Sim、OmniDrones、TorchRL 和 TensorDict。下面使用 Isaac Sim 自带的 python.sh，并安装项目内的 editable 源包。普通 Colab pip 环境无法替代 Isaac Sim 的 omni.* 模块。

In [ ]:
ISAACSIM_PATH = ISAACSIM_PATH.expanduser().resolve()
ISAAC_PYTHON = ISAACSIM_PATH / "python.sh"
if not ISAAC_PYTHON.is_file():
    raise FileNotFoundError(f"找不到 {ISAAC_PYTHON}；请修改 ISAACSIM_PATH。")
TRAIN_PYTHON = str(ISAAC_PYTHON)

orbit_sim_link = TRAIN_ROOT / "third_party" / "orbit" / "_isaac_sim"
orbit_sim_link.parent.mkdir(parents=True, exist_ok=True)
if not orbit_sim_link.exists():
    orbit_sim_link.symlink_to(ISAACSIM_PATH, target_is_directory=True)

run([TRAIN_PYTHON, "-m", "pip", "install", "-q",
     "tensorboard", "wandb", "hydra-core", "einops",
     "numpy==1.26.4", "pydantic<2", "imageio-ffmpeg==0.4.9",
     "moviepy==1.0.3", "pyyaml", "rospkg", "matplotlib"])
for package in [
    TRAIN_ROOT / "third_party" / "OmniDrones",
    TRAIN_ROOT / "third_party" / "tensordict",
    TRAIN_ROOT / "third_party" / "rl",
]:
    run([TRAIN_PYTHON, "-m", "pip", "install", "-e", str(package)])

scripts_dir = TRAIN_ROOT / "training" / "scripts"
os.environ["ISAACSIM_PATH"] = str(ISAACSIM_PATH)
os.environ["PYTHONPATH"] = os.pathsep.join(filter(None, [str(scripts_dir), os.environ.get("PYTHONPATH", "")]))
run([TRAIN_PYTHON, "-c",
     "import torch; from omni.isaac.kit import SimulationApp; "
     "from omni_drones import *; print(torch.__version__)"],
    env=os.environ)

## 3. 启动训练

train.py 会把标量写入 TENSORBOARD_DIR，把 checkpoint_*.pt 和 resolved_config.yaml 写入 CHECKPOINT_DIR。WandB 使用 offline 模式。

In [ ]:
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
TENSORBOARD_DIR.mkdir(parents=True, exist_ok=True)
TRAIN_LOG.parent.mkdir(parents=True, exist_ok=True)
train_command = [
    TRAIN_PYTHON, str(TRAIN_SCRIPT), "headless=True",
    f"device={DEVICE}", f"env.num_envs={NUM_ENVS}",
    f"max_frame_num={MAX_FRAME_NUM}",
    f"eval_interval={EVAL_INTERVAL}", f"save_interval={SAVE_INTERVAL}",
    "wandb.mode=offline", "tensorboard.enabled=True",
    f"tensorboard.log_dir={TENSORBOARD_DIR.as_posix()}",
    f"checkpoint_dir={CHECKPOINT_DIR.as_posix()}",
    "hydra.job.chdir=False",
]
if RUN_TRAINING:
    with TRAIN_LOG.open("w", encoding="utf-8") as log_file:
        result = subprocess.run(train_command, cwd=TRAIN_ROOT, env=os.environ,
                                stdout=log_file, stderr=subprocess.STDOUT, check=False)
    print(TRAIN_LOG.read_text(encoding="utf-8", errors="replace")[-4000:])
    if result.returncode != 0:
        raise subprocess.CalledProcessError(result.returncode, train_command)
else:
    print("RUN_TRAINING=False：已跳过训练。")

## 4. 查看 TensorBoard

训练完成或训练进行中都可以执行下面的单元；event 文件会持续更新。

In [ ]:
%load_ext tensorboard
%tensorboard --logdir $TENSORBOARD_DIR

## 5. 打包并下载模型、TensorBoard 日志和训练日志

In [ ]:
sys.path.insert(0, str(WORKSPACE))
from tools.package_artifacts import package_artifacts
archive = package_artifacts(
    output=ARTIFACT_ARCHIVE,
    checkpoint_dir=CHECKPOINT_DIR,
    tensorboard_dir=TENSORBOARD_DIR,
    log_file=TRAIN_LOG if TRAIN_LOG.exists() else None,
)
print("Archive:", archive)
try:
    from google.colab import files
    files.download(str(archive))
except ImportError:
    print("不在 Colab 中；请手动下载：", archive)

## 6. 把每次代码修改同步到 GitHub

请先在 Colab 配置 GitHub SSH key 或 HTTPS credential helper，再把 PUSH_CHANGES 改为 True。脚本不会 force-push，训练输出由 .gitignore 排除。

In [ ]:
sync_script = WORKSPACE / "tools" / "sync_to_github.py"
sync_command = [
    sys.executable, str(sync_script), "--repo-root", str(WORKSPACE),
    "--remote", "origin", "--branch", BRANCH,
    "--message", "Update NavRL training workflow",
]
if PUSH_CHANGES:
    run(sync_command, env=os.environ)
else:
    print("未推送；确认凭据后把 PUSH_CHANGES 改为 True。")
    print("将执行：", " ".join(sync_command))

## 7. 在本地验证下载的模型

在本地准备与项目匹配的 Isaac Sim/依赖，解压 zip 后从仓库根目录运行：

python tools/validate_checkpoint.py --checkpoint /path/to/unzipped/checkpoints/checkpoint_final.pt --max-frames 4096 --device cuda:0

如需缩小验证环境，可追加 --num-envs 2 --override env.num_obstacles=0。脚本会调用原有 eval.py，并通过 checkpoint=... 传入模型路径。